# Research experiment (development period only)

Parameterized with Papermill: `config_json` holds overrides of `quantlab.experiment.DEFAULT_CONFIG`.
This notebook can only load **research-mode** data (prices through 2016-03-31; signals through 2015-12-31). Launch many experiments with `python scripts/run_research.py experiments/<plan>.json`.

In [ ]:
config_json = "{}"  # overrides, e.g. {"name": "ridge_all", "features": [...]}

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'quantlab').is_dir())
sys.path.insert(0, str(root))
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
from quantlab.experiment import make_config, run_research
from quantlab import features as F
cfg = make_config(**json.loads(config_json))
print(json.dumps(cfg, indent=1))

## Run: walk-forward validation folds + development backtest (with accounting audit)

In [ ]:
art = run_research(cfg)
s = art['summary']
display(Markdown(Path(root / s['output_dir'] / 'summary.md').read_text()))

## Validation folds (one fit per year at its first signal; training labels completed before the fit)

In [ ]:
display(art['folds'].round(4))

## Development backtest vs S&P 500 TR

In [ ]:
display(art['performance'])
display(art['relative'].round(3))
display(art['diagnostics'].round(4))
display(Image(filename=str(root / s['output_dir'] / 'dev_cumulative_return.png')))

## Accounting and timing audit (independent recomputation)

In [ ]:
display(art['audit'])
assert art['audit'].passed.all()

## Walk-forward fit audit

In [ ]:
display(art['fit_audit'])